# Argument and research validation
Synthetic cards exercise the same parser, index builder, research context and query port used by the CLI.

In [1]:
import sys, tempfile, shutil
from pathlib import Path
root = Path.cwd()
while not (root / "knowledge_palace").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from knowledge_palace.tests.test_research import ARGUMENT, SYNTHESIS, MINI
from knowledge_palace.graph.builder import write_index, ensure_index
from knowledge_palace.graph.port import GraphQueryPort
from knowledge_palace.interaction.research_helpers import research_context, render_progress
temp = tempfile.TemporaryDirectory()
vault, state = Path(temp.name)/"vault", Path(temp.name)/"state"
shutil.copytree(MINI, vault)
paper = vault/"papers/alpha-2020-echo.md"
paper.write_text(paper.read_text(encoding="utf-8") + ARGUMENT, encoding="utf-8")
brief = vault / "briefs/2026-07-13-gaps.md"
brief.write_text(brief.read_text(encoding="utf-8") + SYNTHESIS, encoding="utf-8")
_, document = write_index(vault, state)
document["payload"]["identity_report"]

{'works': 3,
 'claims': 4,
 'gaps': 2,
 'bridges': 1,
 'briefs': 1,
 'confirmations': [],
 'parse_errors': [],
 'unresolved_refs': []}

In [2]:
context = research_context(document["payload"], "rare-window-token")
context["works"], context["relations"]

({'work:alpha-2020-echo': {'id': 'work:alpha-2020-echo',
   'kind': 'work',
   'label': 'Echo Cancellation in Alpha Systems',
   'canonical_ref': {'root': 'vault',
    'path': 'papers/alpha-2020-echo.md',
    'anchor': None},
   'attrs': {'year': '2020',
    'argument': [{'role': 'problem',
      'claim': 'C1',
      'paraphrase': 'Echo has a finite sampling window.',
      'attribution': 'author',
      'scope': 'synthetic observations'},
     {'role': 'advance',
      'claim': 'C1',
      'paraphrase': 'Sampling resolves echo.',
      'attribution': 'system',
      'scope': 'synthetic observations'}],
    'conditions': [{'dimension': 'sampling',
      'value': 'rare-window-token',
      'evidence': 'C1'}],
    'publication_status': 'not-recorded',
    'read_depth': 'full',
    'source_coverage': 'not-recorded'}},
  'work:beta-2021-foxtrot': {'id': 'work:beta-2021-foxtrot',
   'kind': 'work',
   'label': 'Foxtrot Thresholds in Beta Regimes',
   'canonical_ref': {'root': 'vault',
    '

In [3]:
response = GraphQueryPort(vault, state).query_context("claim:alpha-2020-echo#C1")
assert response["node"]["id"] == "claim:alpha-2020-echo#C1"
assert context["relations"][0]["attrs"]["claim_ref"] == "claim:alpha-2020-echo#C1"
print(render_progress(context))


# Research progress: rare-window-token

## Evidence coverage

Coverage: candidate-set-only; 6 candidate cards. This view describes recorded evidence.

- Query `rare-window-token`: 5 index matches before selection.

## Current understanding

- [brief:2026-07-13-gaps](../briefs/2026-07-13-gaps.md)
  [S] Echo has restricted support. Dependencies: claim:alpha-2020-echo#C1, gap:gap-echo-noise. Scope is one window.

## Problem evolution

- [author; 2020] Echo has a finite sampling window. (work:alpha-2020-echo). Scope: synthetic observations. [C1](../papers/alpha-2020-echo.md#C1) — §2 [¶1] / p.2
  Quote: "Echo cancellation improves by delta."

## Recorded advances

- [system; 2020] Sampling resolves echo. (work:alpha-2020-echo). Scope: synthetic observations. [C1](../papers/alpha-2020-echo.md#C1) — §2 [¶1] / p.2
  Quote: "Echo cancellation improves by delta."

## Remaining author questions

No statements recorded for this role in the retrieved candidates.

## Relations and disputes

- `claim

In [4]:
context = research_context(document["payload"], "echo", ["rare-window-token"])
assert len(context["attempts"]) == 2
assert context["current_syntheses"]
assert len(context["candidates"]) <= 15
context["current_syntheses"], context["unread_dependencies"], context["bridges"]


({'brief:2026-07-13-gaps': {'id': 'brief:2026-07-13-gaps',
   'kind': 'brief',
   'label': '2026-07-13-gaps',
   'canonical_ref': {'root': 'vault',
    'path': 'briefs/2026-07-13-gaps.md',
    'anchor': None},
   'attrs': {'role': 'view',
    'evidence_capable': False,
    'synthesis': [{'id': 'S1',
      'statement': 'Echo has restricted support.',
      'claims': 'claim:alpha-2020-echo#C1',
      'gaps': 'gap:gap-echo-noise',
      'rationale': 'Scope is one window.'}],
    'topic': '',
    'date': '2026-07-13',
    'view': ''}}},
 [],
 {'cards': [{'node_id': 'bridge:bridge-echo-to-gamma',
    'canonical_ref': {'root': 'vault',
     'path': 'bridges/bridge-echo-to-gamma.md',
     'anchor': None},
    'status': 'candidate'}],
  'candidates': []})

## Bounded reading plan
`research ask` prints this view: every candidate, a few quoted Claims per work, query matches first.

In [5]:
from knowledge_palace.interaction.research_helpers import bounded_context
view = bounded_context(research_context(document["payload"], "noise floors"), 1)
quoted = view["works"]["work:alpha-2020-echo"]
assert quoted["claims_total"] == 2 and [claim["id"] for claim in quoted["claims"]] == ["C2"]  # the Claim the query matched
assert view["relations"] == []  # C1's relation is left to the card, with C1
view["candidates"], quoted, view["gaps"]["gap:gap-echo-noise"]["statement"]

([{'node_id': 'brief:2026-07-13-gaps',
   'path': 'briefs/2026-07-13-gaps.md',
   'why': ['current-synthesis']},
  {'node_id': 'work:alpha-2020-echo',
   'path': 'papers/alpha-2020-echo.md',
   'why': ['query-match', 'synthesis-dependency']},
  {'node_id': 'gap:gap-echo-noise',
   'path': 'gaps/gap-echo-noise.md',
   'why': ['term:noise', 'query-match']}],
 {'title': 'Echo Cancellation in Alpha Systems',
  'year': '2020',
  'read_depth': 'full',
  'claims_total': 2,
  'claims': [{'id': 'C2',
    'quote': '"Noise floors remain unresolved."',
    'anchor': '§5 / p.9'}]},
 'Echo noise floor remains unresolved in alpha systems.')

## Analysis views
`research source`, `bridges`, `novelty` and `partition` read the same payload and Vault identity.

In [6]:
from knowledge_palace.graph.identity import domain_partition_report, load_vault_identity
from knowledge_palace.interaction.novelty import novelty_pipeline
from knowledge_palace.interaction.project_source import resolve
from knowledge_palace.interaction.research import reference_from_text
from knowledge_palace.semantic.corridors import find_corridors
identity = load_vault_identity(vault)
references = [reference_from_text(text) for text in ("arXiv:2101.00001v3", "https://doi.org/10.99999/Fixture.Alpha2020", "An Unread Paper")]
references, [resolve(reference, identity).get("work", "project-source") for reference in references]

([{'ids': {'arxiv': '2101.00001'}},
  {'ids': {'doi': '10.99999/fixture.alpha2020'}},
  {'title': 'An Unread Paper'}],
 ['work:beta-2021-foxtrot', 'work:alpha-2020-echo', 'project-source'])

In [7]:
corridors = find_corridors(document["payload"], "alpha-domain", "beta-domain")
drill = novelty_pipeline(document["payload"], "echo drift")
corridors["total"], [entry["kind"] for entry in corridors["corridors"]], drill["corridors"]["total"], drill["works"]["total"]

(2, ['bridge', 'shared-axis'], 2, 2)

In [8]:
report = domain_partition_report(identity)
sorted(report["roots"]), dict(report["root_pairs"]), report["cards_without_root"]

(['alpha-domain', 'beta-domain'], {('alpha-domain', 'beta-domain'): 1}, [])

In [9]:
temp.cleanup()